In [1]:
import pandas as pd
coords = pd.read_csv("../data/coordinates.csv.gz")
coords.head()

,root_id,position,supervoxel_id
0,720575940613726179,[352484 175164 229040],76704474512672967
1,720575940623219453,[631424 216180 55880],81490235707935665
2,720575940613998790,[713412 223496 57080],82897679310436225
3,720575940606611401,[444928 283372 77240],78254373054459056
4,720575940620628957,[516764 103356 67880],79517986792372064


# 3D visualization of test predictions

## Φόρτωση θέσεων

Το `coordinates.csv.gz` (FlyWire v783, από το Codex) έχει μία θέση ανά **supervoxel**, όχι ανά νευρώνα — γι' αυτό υπάρχουν πολλαπλές γραμμές με το ίδιο `root_id`. Η `position` είναι string της μορφής `"[x y z]"`, σε voxel units.

In [2]:
type(coords["position"].iloc[0])

str

In [3]:
print(coords.shape)
print(coords["root_id"].nunique())

(238909, 3)
139255


## Ένα σημείο ανά νευρώνα

Ελέγχω πόσες γραμμές και πόσα μοναδικά `root_id` υπάρχουν, και κρατάω μία γραμμή ανά νευρώνα (την πρώτη supervoxel θέση — αρκεί ως προσεγγιστική θέση του νευρώνα στον χώρο).

In [4]:
coords = coords.drop_duplicates(subset="root_id")
coords.shape

(139255, 3)

In [5]:
xyz = coords["position"].str.strip("[]").str.split(expand=True).astype(float)
xyz.columns = ["x", "y", "z"]
coords = coords[["root_id"]].join(xyz)
coords.head()

,root_id,x,y,z
0,720575940613726179,352484.0,175164.0,229040.0
1,720575940623219453,631424.0,216180.0,55880.0
2,720575940613998790,713412.0,223496.0,57080.0
3,720575940606611401,444928.0,283372.0,77240.0
4,720575940620628957,516764.0,103356.0,67880.0


## Σπάσιμο του position σε x/y/z

Η στήλη `position` είναι string (`"[x y z]"`), όχι αριθμοί. Την καθαρίζω από τις αγκύλες, τη σπάω σε τρεις στήλες και τις μετατρέπω σε float, ώστε να μπορούν να χρησιμοποιηθούν ως συντεταγμένες στο γράφημα.

In [6]:
preds = pd.read_csv("../results/test_predictions.csv")
df = preds.merge(coords, on="root_id", how="left")
print(df.shape)
print(df[["x", "y", "z"]].isna().sum())
df.head()

(20887, 6)
x    0
y    0
z    0
dtype: int64


,root_id,y_true,y_pred,x,y,z
0,720575940633222983,optic,optic,710356.0,321908.0,99320.0
1,720575940629512186,central,central,478460.0,89780.0,137400.0
2,720575940622078934,visual_projection,visual_projection,707120.0,290264.0,240440.0
3,720575940609667083,optic,optic,773888.0,368276.0,159800.0
4,720575940630990155,optic,optic,270764.0,280620.0,253840.0


## Merge με τις προβλέψεις

Ενώνω τις θέσεις με το `results/test_predictions.csv` (μόνο οι 20.887 νευρώνες του test set, με `y_true` και `y_pred` του `model_np`). Ελέγχω ότι δεν υπάρχουν κενές συντεταγμένες μετά το merge.

In [7]:
import plotly.express as px

fig = px.scatter_3d(
    df, x="x", y="y", z="z",
    color="y_pred",
    opacity=0.6,
)
fig.update_traces(marker_size=1.5)
fig.update_layout(
    scene_aspectmode="data",
    legend_itemsizing="constant",
)
fig.show()

## Γράφημα ανά κλάση

3D scatter των 20.887 νευρώνων του test set, στη θέση τους στον εγκέφαλο, χρωματισμένοι ανά προβλεπόμενο `super_class` (`y_pred`).

In [8]:
df["correct"] = df["y_true"] == df["y_pred"]

fig2 = px.scatter_3d(
    df, x="x", y="y", z="z",
    color="correct",
    color_discrete_map={True: "lightgray", False: "red"},
    opacity=0.5,
    hover_data=["y_true", "y_pred"],
)
fig2.update_traces(marker_size=1.5)
fig2.update_layout(scene_aspectmode="data", legend_itemsizing="constant")
fig2.show()

## Γράφημα σωστό/λάθος

Το ίδιο γράφημα, αλλά χρωματισμένο μόνο ανά `y_true == y_pred`, ώστε να φαίνεται *πού* στον εγκέφαλο κάνει λάθος το μοντέλο, ανεξάρτητα από ποια κλάση προς ποια.

In [9]:
fig.write_html("../results/brain3d_predictions.html")
fig2.write_html("../results/brain3d_errors.html")

## Αποθήκευση

Τα διαδραστικά `.html` (περιστροφή, zoom, hover ανά νευρώνα) και τα στατικά `.png` για το README.

In [10]:
fig.write_image("../results/brain3d_predictions.png", scale=2)
fig2.write_image("../results/brain3d_errors.png", scale=2)

## Παρατήρηση

Τα λάθη είναι συγκεντρωμένα στον κεντρικό εγκέφαλο, όπου συνυπάρχουν κλάσεις με παρόμοιο χωρικό προφίλ (π.χ. `central` που προβλέφθηκε ως `ascending`), ενώ οι οπτικοί λοβοί είναι σχεδόν χωρίς λάθη.